# Numerical Stability

| Difficulty | █████░░░░░ 5/10 |
| :--- | :--- |
| Prerequisites | 06.01 |
| Time | ~65 min |
| Colab GPU | Not needed |

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/NiravRVaghasiya/maths-for-ai/blob/main/06_Numerical_Methods/02_numerical_stability.ipynb)

---

## 🎯 Learning Objective

Learn catastrophic cancellation and the log-sum-exp trick, and derive a numerically stable softmax and cross-entropy implementation.

---

## 📐 Theory

`06.01` established that floats are finite approximations. This notebook covers what happens
when an otherwise-correct *formula* gets evaluated using those approximations in the wrong
order — and how to reorder the arithmetic so the same mathematical answer survives intact.

### Catastrophic cancellation

Subtracting two nearly-equal large numbers can destroy almost all of the result's meaningful
digits. Consider $(10^{20} + 1) - 10^{20}$: mathematically this is exactly $1$. But in floating
point, $10^{20} + 1$ first rounds to the nearest representable float32/float64 value — and
$1$ is *far* smaller than $10^{20}$'s machine epsilon-scaled gap between representable numbers
(recall `06.01`), so the addition doesn't move the stored value at all. Subtracting $10^{20}$
back out then returns $0$, not $1$ — the "+1" was lost the instant it was added, not during the
subtraction. **Catastrophic cancellation** is this general phenomenon: whenever a computation
subtracts two nearly-equal quantities, whatever rounding error was already present in each one
individually gets massively amplified in *relative* terms, because the (small) true difference
is now being measured against a (large) pre-existing rounding error.

### The log-sum-exp trick

A recurring pattern in probability and ML is computing $\log\sum_i e^{x_i}$ — the normalizing
constant behind softmax, log-likelihoods, and more. Evaluated naively, $e^{x_i}$ overflows to
`inf` the moment any $x_i$ exceeds about $709$ (where $e^{709}$ hits float64's max), even though
the final *logarithm* of that sum might be a perfectly ordinary, modest number. The fix uses
one algebraic identity:

$$\log\sum_i e^{x_i} = m + \log\sum_i e^{x_i - m}, \qquad m = \max_i x_i$$

Subtracting the max *before* exponentiating guarantees every exponent argument is $\le 0$, so
every $e^{x_i-m} \in (0, 1]$ — no overflow is possible, and the identity is exact algebra, not
an approximation (you can verify it by factoring $e^m$ out of the sum and taking the log). This
is the **log-sum-exp trick**, and it's one of the most-used stability tricks in all of applied
probability and machine learning.

### Numerically stable softmax

Softmax, $\text{softmax}(x)_i = e^{x_i} / \sum_j e^{x_j}$, has the exact same overflow problem
as log-sum-exp, since it's built from the same $e^{x_i}$ terms. The fix is the identical
subtract-the-max trick:

$$\text{softmax}(x)_i = \frac{e^{x_i-m}}{\sum_j e^{x_j-m}}, \qquad m=\max_j x_j$$

This is provably the *same* function as the naive formula — multiplying every numerator and
denominator term by the constant $e^{-m}$ doesn't change the ratio — but it's numerically safe:
the largest exponent argument is now exactly $0$ (giving $e^0=1$, never overflowing), and every
other argument is $\le 0$.

### Numerically stable cross-entropy

Cross-entropy loss (recall `05.03`) combines a $\log$ with a softmax:
$-\log(\text{softmax}(x)_y)$ for the true class $y$. Computed as two separate steps (softmax,
then log), this risks a second failure mode: if softmax legitimately produces a very small
probability (a confident wrong prediction), taking its log can lose precision or even hit
$\log(0)=-\infty$. The fix is to never materialize the intermediate softmax probabilities at
all — combine the softmax and the log analytically into a single **log-softmax** formula built
directly from log-sum-exp:

$$\log\text{softmax}(x)_y = x_y - \log\sum_j e^{x_j} = x_y - \left(m + \log\sum_j e^{x_j-m}\right)$$

This is exactly what `torch.nn.functional.cross_entropy` and `log_softmax` compute internally —
never forming the (potentially tiny, precision-losing) intermediate probability at all.

---

In [ ]:
# Setup
import numpy as np
import matplotlib.pyplot as plt
%matplotlib inline
plt.style.use('seaborn-v0_8-whitegrid')

## 👁️ Visual Intuition

Plotting naive vs. stable softmax's output as the input magnitude grows shows the naive
version collapsing into NaN well before the stable version shows any sign of trouble.

In [ ]:
# As we scale up the SAME 3-logit pattern (only relative differences matter for softmax's
# true output), naive softmax breaks down long before stable softmax does.
def naive_softmax(x):
    e = np.exp(x)
    return e / e.sum()

def stable_softmax(x):
    shifted = x - np.max(x)
    e = np.exp(shifted)
    return e / e.sum()

base_logits = np.array([1.0, 2.0, 3.0])  # relative shape stays fixed; only overall SCALE grows
scales = np.linspace(1, 300, 80)
naive_p0, stable_p0, naive_has_nan = [], [], []
for s in scales:
    logits = base_logits * s
    with np.errstate(over='ignore', invalid='ignore'):
        naive_result = naive_softmax(logits)
    naive_p0.append(naive_result[0])
    naive_has_nan.append(np.any(np.isnan(naive_result)))
    stable_p0.append(stable_softmax(logits)[0])
naive_has_nan = np.array(naive_has_nan)

fig, ax = plt.subplots(figsize=(8, 5))
ax.plot(scales, stable_p0, color="#4C72B0", lw=2.5, label="stable softmax (subtract max first)")
ax.plot(scales, naive_p0, color="#C44E52", lw=1.5, ls="--", label="naive softmax (exp then normalize)")
if naive_has_nan.any():
    ax.axvline(scales[naive_has_nan.argmax()], color="gray", ls=":", label="naive first produces NaN here")
ax.set_xlabel("scale factor applied to logits [1,2,3]"); ax.set_ylabel("softmax output, class 0")
ax.set_title("Naive softmax silently returns 0 (then NaN) as exp() overflows;\nstable softmax never does")
ax.legend()
plt.show()

if naive_has_nan.any():
    first_nan_scale = scales[naive_has_nan.argmax()]
    print(f"Naive softmax first produces a NaN entry around scale factor ~ {first_nan_scale:.1f}")
else:
    print("Naive softmax did not hit NaN in this range, but see the numbers below --")
print(f"even BEFORE that point, naive softmax already returns exactly 0.0 for class 0 well before")
print(f"scale={scales[naive_has_nan.argmax()] if naive_has_nan.any() else scales[-1]:.0f}, which is")
print(f"arguably worse than a visible NaN: a silently WRONG (but plausible-looking) probability.")
print(f"Stable softmax's output stays well-defined and correctly proportioned at every scale tested.")

## 🐍 Implementation from Scratch

We reproduce catastrophic cancellation with real numbers, implement naive and stable versions
of softmax and log-sum-exp side by side, and confirm the stable versions match `scipy.special.logsumexp`
and never produce NaN/inf where the naive versions do.

In [ ]:
from scipy.special import logsumexp as scipy_logsumexp

# --- Catastrophic cancellation: a "+1" that gets silently dropped ---
huge = 1e20
result = (huge + 1.0) - huge
print(f"(1e20 + 1) - 1e20 = {result}  (mathematically exact answer is 1.0)")
print(f"The '+1' was lost the instant it was added -- 1e20's rounding gap (recall 06.01) is far")
print(f"bigger than 1.0, so 1e20 + 1.0 rounds right back down to 1e20 before any subtraction happens.")

# --- Naive vs stable softmax ---
def naive_softmax(x):
    e = np.exp(x)
    return e / e.sum()

def stable_softmax(x):
    shifted = x - np.max(x)
    e = np.exp(shifted)
    return e / e.sum()

x_normal = np.array([1.0, 2.0, 3.0])
x_large = np.array([1000.0, 1001.0, 1002.0])   # same RELATIVE differences, much larger scale

print(f"\nOn modest inputs {x_normal}:")
print(f"  naive:  {naive_softmax(x_normal)}")
print(f"  stable: {stable_softmax(x_normal)}")
print(f"  agree: {np.allclose(naive_softmax(x_normal), stable_softmax(x_normal))}")

with np.errstate(over='ignore', invalid='ignore'):
    naive_result = naive_softmax(x_large)
print(f"\nOn large inputs {x_large}:")
print(f"  naive:  {naive_result}  <- NaN, because exp(1000) already overflowed to inf")
print(f"  stable: {stable_softmax(x_large)}  <- still a valid probability distribution")

# --- Naive vs stable log-sum-exp, cross-checked against scipy's implementation ---
def stable_logsumexp(x):
    m = np.max(x)
    return m + np.log(np.sum(np.exp(x - m)))

with np.errstate(over='ignore'):
    naive_lse = np.log(np.sum(np.exp(x_large)))
stable_lse = stable_logsumexp(x_large)
scipy_lse = scipy_logsumexp(x_large)
print(f"\nlog-sum-exp on {x_large}:")
print(f"  naive:  {naive_lse}  <- infinite, exp() overflowed before the log could rescue it")
print(f"  stable: {stable_lse:.6f}")
print(f"  scipy:  {scipy_lse:.6f}")
print(f"  stable matches scipy: {np.isclose(stable_lse, scipy_lse)}")

## 🤖 Why This Matters for AI

Every production cross-entropy loss implementation — `torch.nn.functional.cross_entropy`,
`torch.nn.CrossEntropyLoss`, TensorFlow's `softmax_cross_entropy_with_logits` — uses the
log-sum-exp trick internally, computing a fused log-softmax rather than softmax followed by a
separate log. A model that produces a single unusually large logit (common early in training,
or with an unstable learning rate, recall `04.06`) would otherwise crash training outright with
a NaN loss, rather than simply reporting a large-but-finite one. Below, we implement a
numerically stable cross-entropy from scratch, and stress-test it against `torch.nn.functional.cross_entropy`
on logits large enough to make the naive two-step version fail outright.

In [ ]:
import torch

def stable_log_softmax(x):
    # This is the fused log-softmax mentioned in the Theory section: never materializes
    # an intermediate probability, so it can't underflow OR overflow independently.
    m = np.max(x, axis=-1, keepdims=True)
    return x - m - np.log(np.sum(np.exp(x - m), axis=-1, keepdims=True))

def stable_cross_entropy(logits, target_idx):
    log_probs = stable_log_softmax(logits)
    return -log_probs[np.arange(len(logits)), target_idx]

rng = np.random.default_rng(0)
# Deliberately huge logits -- exactly the kind of spike an unstable training run could produce
logits = rng.normal(scale=500, size=(4, 5))
targets = np.array([0, 2, 4, 1])

ours = stable_cross_entropy(logits, targets)
torch_ce = torch.nn.functional.cross_entropy(
    torch.tensor(logits), torch.tensor(targets), reduction='none'
).numpy()

print(f"Logits scale: {np.abs(logits).max():.1f} (would make naive exp() overflow instantly)")
print(f"\nOur from-scratch stable cross-entropy: {np.round(ours, 4)}")
print(f"torch.nn.functional.cross_entropy:      {np.round(torch_ce, 4)}")
print(f"Match: {np.allclose(ours, torch_ce, atol=1e-4)}")
print(f"Any NaN in our result: {np.any(np.isnan(ours))}")

# Contrast: the naive two-step version DOES fail on these same logits
with np.errstate(over='ignore', invalid='ignore'):
    naive_probs = np.exp(logits) / np.exp(logits).sum(axis=-1, keepdims=True)
    naive_ce = -np.log(naive_probs[np.arange(len(logits)), targets])
print(f"\nNaive (unstable) cross-entropy on the SAME logits: {naive_ce}")
print("The naive version fails exactly where the stable one succeeds -- same math, different arithmetic.")

## 🏋️ Exercises

### Warm-up
1. Derive the log-sum-exp identity $\log\sum_i e^{x_i} = m + \log\sum_i e^{x_i-m}$ by hand
   (factor $e^m$ out of the sum before taking the log). Verify numerically with
   `stable_logsumexp` on $x=(3,5,1)$.

### Practice
2. Modify `naive_softmax` to add a small guard (`np.clip(x, -50, 50)` before exponentiating)
   instead of using the max-subtraction trick. Test it on `x_large` from the Implementation
   cell. Does clipping recover a valid probability distribution? Explain why clipping is a
   different (and generally worse) fix than the log-sum-exp trick -- what information does
   clipping destroy that max-subtraction preserves?

### Challenge
3. Implement a numerically stable version of the **sigmoid** function,
   $\sigma(x) = 1/(1+e^{-x})$, that avoids overflow for large negative $x$ (where $e^{-x}$
   would overflow) using a case split (different formulas for $x\ge 0$ and $x<0$, both
   algebraically equal to the original). Verify it matches `scipy.special.expit` across a wide
   range of inputs including $x=-1000$ and $x=1000$, where the naive formula fails.

---

## 📚 Further Reading
- Goodfellow, Bengio & Courville, *Deep Learning*, Ch. 4.1 (Overflow and Underflow)
- Blanchard, Higham & Higham, ["Accurately Computing the Log-Sum-Exp and Softmax Functions"](https://arxiv.org/abs/1909.03469)

---

*Next notebook: [Numerical Differentiation](03_numerical_differentiation.ipynb)*